# Semantic decoder scheduler: real-cache component run

Canonical project: SemanticSchedulerEnd2End. Submitted from WSL user Satya, Kaggle account satyapaladuguplus. CPU-only; no GPU/teacher/extraction/model training. This first version implements full93 lossless transport and dynamic FIFO/RR/random/SPI-input baseline replay. It is not the full temporal P/D/RL comparison.

Input: completed image-pool v5 output, archive SHA verified before use. Frozen source is embedded below from the canonical repository; source hashes, actual environment and all computational cell logs are persisted. No local Windows environment transfers to Kaggle. Managed Kaggle Python and its standard library suffice; no pip installs.

Scope: select one ready message, arrivals continue during unit synthetic nonpreemptive service; MIXED_IID images, no retries. Input-SPI delay is descriptive and not independent urgency evaluation. Relevance/novelty belong to streaming extraction and do not block these baseline policies. Qualified contextual teacher training remains separate.


In [ ]:
import sys,os,json,time,platform,contextlib,traceback,datetime,hashlib,base64,zipfile,io,subprocess,shutil
from pathlib import Path
RUN=Path('/kaggle/working/experiments/stream_scheduler/runs')/datetime.datetime.now(datetime.timezone.utc).strftime('compact-real-cache-%Y%m%dT%H%M%SZ')
RUN.mkdir(parents=True,exist_ok=False)
(RUN/'logs').mkdir()
STATUS=[]
class Tee:
    def __init__(self,a,b):self.a,self.b=a,b
    def write(self,s):self.a.write(s);self.b.write(s);self.b.flush();return len(s)
    def flush(self):self.a.flush();self.b.flush()
@contextlib.contextmanager
def stage(name):
    start=time.perf_counter();row={'stage':name,'status':'running','started_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()};STATUS.append(row)
    (RUN/'cell_status.json').write_text(json.dumps(STATUS,indent=2))
    with (RUN/'logs'/f'{name}.log').open('w',buffering=1) as log:
        with contextlib.redirect_stdout(Tee(sys.stdout,log)),contextlib.redirect_stderr(Tee(sys.stderr,log)):
            print('START',name,'output',RUN,flush=True)
            try:
                yield
                row['status']='complete'
            except BaseException as e:
                row['status']='failed';row['error']=type(e).__name__+': '+str(e);traceback.print_exc();raise
            finally:
                row['elapsed_seconds']=time.perf_counter()-start
                (RUN/'cell_status.json').write_text(json.dumps(STATUS,indent=2))
                print('END',name,row['status'],row['elapsed_seconds'],flush=True)
with stage('01_environment'):
    info={'interpreter':sys.executable,'python':sys.version,'platform':platform.platform(),'cpu_count':os.cpu_count(),'execution_target':'Kaggle managed CPU; submitted from WSL', 'gpu_requested':False,'dependencies':'Python standard library; no notebook installations','run_dir':str(RUN)}
    (RUN/'environment.json').write_text(json.dumps(info,indent=2));print(json.dumps(info,indent=2),flush=True)


In [ ]:
with stage('02_source_snapshot'):
    blob=base64.b64decode('UEsDBBQAAAAIAPt7Xl3BLQZbLAUAACkOAAAlAAAARXZhbHVhdGlvbi9jb21wYWN0X3NjZW5lX3RyYW5zcG9ydC5weZVW32/bNhB+91/B5aGWEsVO7NiBjbQP2TIkwNAN7VAMCAyDkU42F5n0SMqJi2F/++5Iyfphp27zEpPi3X13991Hnpyc/KaMycAYZjWXZq20ZSplJgYJ51xrvj3fDNgKuMk1rEBaw3KZgGacrTUIaSzPMkhwoVKRQa/zUTHNX9gC1Aqs3jINsdqA3kYsA64lHk0FZAnur9SGZxFT6Cu3S3QtYm6FkkwYJlbrTEDS65ycnHRwQbCM1XlsO51OnHHE+7NarXlsPxPSP0vs0w7Dvy93nz4//P6RvWdDt/7j092vD3/h0rvofXb/gu7N7e19N+y4MwmkbD4XUtj5PDCQpVGZ01wkEcJdCLTezs2SD0Zj9zHJY9C7jVjl0s75GoHH/CmD0GOhP5Eyu11DUDkMKUmpLBPSUgXo5wW7eV+LSavxaDQcVX7oT3NhgH3hWQ53WisddHP0cTnG+DIVC+zSrhns4RfE/U8ucA/TLD1Qcr1anHrQ5qEyaTzytLVgeqlWqyW8Bq1q7Pt2pdkza9WsZVaVDg1tvs4g2KtpvaQZyKABM2Q/YccHVM/dtzLkoW81v/R1MjxW6TTPMrbkZgmGcZkweEUCTobnnpBl1Q+UHNFyuQ0cCV53vX9SKmMpInpFGrRrEB4DQ9bAJStNRCbsth68ojXOmUi4hYLXL3igyU4qidt1VRpPqEy0fryY0c5gt76csXfsv6OFEtKFZEblOgavJgxVwOB49434Cv004wvTrJALMJgOr1zMJgHL+MOr6XhcfS+7ewyPB1Ce7m8U1ivPODoum7YSZsVtvKxBcuQzlWrkEvXmeU5kRumYDO+7vpQRG9eozDVqJRr5bEaUz2hW+VzhJ5xXNxJzNx1BkdhoOhxdzyLWxT7aDGpA+JtGo+spduugERYUg334gLR2Elv8PlYpqeRX0IqtuHlma54kQi5qXomtImKBjHhIpAWZr0ATtb6KtR9XE7kShC0Cx6ihJAjI2gCBvWPBJbu5YSIMI7/JG5sNW8rFDRxNTWtOHsWsFNDiTLnUbsmbML7NV5dAa6LQWy75hovMSdOGzOrMLQBSxGpELl3sICZr7QC5JoTfAybFUCjiCViILVrB61oZVBzYiARkXG+yBptrWTA18tyr5h4Pq+SNoW+YsGKcKqFwp3eHMayB8pC/THs0C8EwKnOO2tdKyM78t/F4WvEfb3ktgAI+BiJyLPKU+i5GuTq7QlYezZprBy44gK5syL/sKjqAz4/SI6lfASychXsdYmUmNKMzMut2e38rgbeIFwYXDB8T97fd6BR8RuBz8U73GlaALi8xtwrZjVu5YocMMjzgflcNTaDWUOwT31dxt4uero4NOkKX+N4CIr17SNF7LhWvNXYVgh0xJ9a791CLCHVZdNHrkAoX7lqhxjlPdIVc06p0WJPzojtH3zzyWaoXWcPuLPvlHXPgeilDtyrTqtvA3X20esR7qFTP6gTdhhPkwNVpceZH5KVoO3pb2GXfcbslJI2xxCczN5Zm5WJ2OsFZI766N3kwGaJsnl82bIl2z0Q7fAwvICjwHZAcN3hv3WuOxWQbDSZnV6fP+/OAFRP0OHXgaHjZh/dFnUgEgzel7q1WGny4Q9JPcie7FvqtetEYbbv7QHyx8ALw+kA/ioJhfpFonH9SybZKuZhYd4efejeVHnixwaWr4Gg6mNT0C6fyG/whml9c/wglvLR6RrRSLCD7PtY0lLSofFoHj4OoJPZwRqCb76az1ov8zHltvrvfUv1Cq2iz8z9QSwMEFAAAAAgA+3teXX028qLjCwAA9xwAAC8AAABleHBlcmltZW50cy9zdHJlYW1fc2NoZWR1bGVyL21pbmlfa2FnZ2xlX3J1bi5weZ1ZeW/byBX/35+CDVAMaY+pI7FjS8sFUm+Kpkh2jcQtWqgCMSJH0li8OkPa1hr+7n1vDh6Wnd2tAFs85t3H/N7ozZs3fymbIuWpJznLThOWbLlXS1aoqpS1x4rUS/cFy0XirZjimSg4rKwytg+9n0tPFHe8qIG65kgpvYyteKbCN2/eHIncsJCbiknFj9ayzL2kzDKe1KIslGcXXIECNZfUS/maNVmdiqTGm/823PHYMrXNxMrdmi94EDa1yNzTW1UWRkjFalzuBFzDrVsElqVl7u7UXrnLmsm1yFqJtcj50dERqOSlTV75yJLesazhwezIgw8+CMEwMD/Md6mQvrlR0Y1sOOUPQtVxudN3QUdxL0XN45o/1D7qGyJz5WvGVEAcijqaUpZl5X1csCL6K8sUD07IfwoSWHWykqVxGyFflmXdU8mLPHwyIh+RJ0NHj5Iyr1hSxyrhBe9Iw2pPNJ2qeAJ0Q6+G+DRGd8bolzgrE83NJ6+wIxTlG1PzMm0yfsjTPDdckb+P/4JWiRBt4zLkDyDarPXNl1kjed3IwnIPr4we31CNG6eFdZJJUXDOvaINjxNMMZpzpdjG3aEsqjhPadk4D8LV7wymLDaRyaXwq/7ykZU1hWOK8zSCRaFieQVmaE1yUfjPlOCFfhUEhvQecqC8j/Cx4xKMUFP9lkkpIKwqUmApT/3FOFweI9MLOqAIThYouinEupS5P6aGbeDBrRdD0WIZbMC37AFeDkhPL4JgaXS5LVcqWjwSp7FIyWxNHtHOp9mjeCLUIyJ3b+Siu1nCG1U2MrGvQBf0lRHqwhFQLcV+iLWNzBgSV0JzxO/lgmT8jmeGK3aFIuFkJp60NYL6krIAbeJFk3PJau7/KqrWIuqcBnZpgdtSil/LIjI+OXk7NfFkxU5Fj+QK6lMkqMeYkr+JzZbMJpR84alocjKbUvK5vCezt082HxX0KyAzt6hPVWYi2aM6C7IG9xNKJDbYWJYrUeCdzha4qCTHTsdTqAQR67XLWeuSihepgBzTbdDHUARzbNHpPlos514BSTIO59BuITGNq6PTydzjdzprcUmyLUvF5QtZenI5CVpB2OiQoH1wv4VyRwE/WE/pTcC3Cnlgo1YjmPXD54k1kNTmlSZw63vXizHE0sV5+SOIGPLADxr24vLBSqPjb4r4IXpRhlYyZBXSOLvCqoQyXdd+ELxuF9gODH+MrGNm3gqe7wbrV3wDbRudGlZcrk2qcxkXyj9gbHIligYZcqiuCW9b9I+3i15xLXXW3eqaRh2fggN6SATFC5clBW49vtJkCskse1RI/djLp4DaN+DTQ6a3EfYd//aZeOQy1C+KBvIDuuP7KGP5KmXe7cy/7YWLIqUr8GVwKLSf7QOug5UAUvq+NfU2A41tRYTwLRJoyTqNv0P6QoXOjN2a9JkluoMsbm3T0sb8PssAL/Hv8P3dfHQlu7T2X07BU52fwWjCz4fEpiYkz8s7aDfPFNRNxXF+PD6+pSQVCrb7BNojVAQl90zU+vJ0oC60wEIoWITGwduTCbW1E1CNIzJec013MvnBldWzFMaX0aRrjCLZwdaKVdDbmHplYLR96nVS6Ro0fnCd3k/sNrhTw4or12sOcY8WvdTG9usyG4MbRZrDsCelkDx3B6RGm98kBiUX5jHsQVYFMsOd2d6Av9yOZp634ob7qP60S+OcsyI2wVFNjrmkb3ru6viMhmxR5/ZOJykA/oK/IK0pOtWslEFosIGCJBO5TrKzrBeqCADYCDCGKcGnECFyRrr3ffhMSHhbQlr1YPStQcrPvd+rE7tjL4yAA2frffa5py0TOrCzXX3aXzLwzR+JgmUw6jFD/9vsecX5zxpUylUiRVVDxKwwyCpnDCB3o0aujBK6WxiB5hLl6atXxf09/onM9HNSpMnGXUuOHoSOBBNBWUE5k59/8XCWwXYB+nuN3MDr/ahqV3rtlZp7Pb2BrGrq0wQavcLkY3vlQQbsickRPYlhirio6QQh9JFgKKA5w380WN4JhIhE7Yt6ywHPeQCFayCXeoQB8GVbDZnZCwiuhaVk1s4LnfUERg5k+OXTvz7+FH/69JMDBZl3J1Jewh4ry6LMys2eoAK6EnC0IIjdIVvg/9NghrGJaMeVnEEiW0TFqsiNzOEHuQFMW9TXeCf9YM6qkKVpzOxzn5yeMplswXWE1vuKRzjrUglbhMBOpOeVl4jAieDo12nsuLFRERBrXZBcoQbwFRry745KZnJ1U1XUI5vjgKqF+rGZLeMAdh5VZrD1BHamVovpcq5gKK8PwZRFUpUUaMy3mw9fb0ws9OlF6unBtHeIcfLaEQah66xR257J28geNYRqy6Zn574by2Cy1iZYb4eQ6DAKyxUJPAZgajbYX5JtU+ywrAVqa7by2TrEPda/OJ6Mp+/0v4CuCAlm27CpUhxbNJkRCLW4Dbf8IRUbrgCU/iki4/VFcj65OF+NL9Nkfc5WF+8vJ5N0ff5uej6ZXk4u375/Pzl7v1pNzi7OxhfTFU8nyeVbuB2fr/k0gYGKCSjsf+JZw0cpS+mTe8jajQebs/BcGhn5V9FLxwydL+xpiXFD3zHaHXXnDhxuYUPEHo0clf9gptAHdE8Nkz6ISGrk5ZMvoEcMnoAg1aMt4rb1emSGSkgKlgNWkMruCbpdPYSqhs7hB8teg99AAsp91Ir0vydFp8rIEZl2ErQhaKfzP0Xn4/H4VQdqHT3TPlxuligNR8O5Er92E5aZWeVwXtUyOp+pCEdf1Iws5wgGfbUgTsfYJCaAK3gIYtImAWznHnY7HagPpO3ea/UZYh33dAErl9GVj/a6ZwFgtWN4ThfJgmjTYgCAsGuyVcbt1pUY3G8Ug1XYtrkC8NupkUDjTKKBoDm03I0oWBat9jUgVjwNglT3ndHxPfShGJ6AOXN72BRpPiHsGPDtOwYDa80K+MMVlgwC59a+ELyuQ+gBDFMJakHlGtd2vHX8HPpFFznmA/G+OJkEf55AmkTReGZ7083XDz9/u/7l6w2h8JqSEaaRxx+AupOpDttQt9FZz8XDFIUdz7mUzBbtjT5tgX7d3tGDtJntYJyjB4kDjydLc6Kyo1U/Y0LoYTk0/mUfDBC30ydlsRabBhGiJfCystw11Rzv74SC7RZHdB1pjyWmg0OL5ljRNc/2eJbhsQb26QIPXnB/9iCRRE7shglREkW0kFo5XThYLuhyfUCUIZCCYU0vs8cEWO18cCyzo37DFS30qd+z06KF/46+u6DhWUD9szEFd9DwEq4hlvRsMqWTcBose+VpY/v14/XnD/8mtON7EMZOlwXEwN8Fy8geTGpt+6ST99PLkx3VMTckZGRo7JmiTkI8ArBbEmC4nEGfeySwS9aN0sGwAxW1hdRLMQ1YNZPAvdUx0aDQIELzdtRbeNSLd0dRXZ4BAa5YoCPCy7NjvyM6nQSQK0MR7AEmQPbQyu9yd9BxXpHncg2KpoFN3vQAt8uvsjLZGZA4975df4JeDnMUA2Q5KmCUzer9CLxQQw6OKrZHV5vjon5y8ocka4Al6SlgYzAz3zCswn5UADCGQqwhdAjwLt/ifMszQN8OYoOW2HARrJQy5+kcqkEpsE15SqOoHioxjKAoMgAulvszDQSqd8iz3grlXV3/w8OAAwQHiC0bABtKd3uoJO96VHAAu9no62dP1U267zNuPRRbDyFSrgGc5KcdZtdAHMB5U7A7JjJs+lg1GmHNtZwC8C2EYbVHiWDZumxkC7AGlgCOrxQOKdCbixSdeojoTjXWew7zbZLbpmfvnrfIO5aJVDeOtjlaMNJ2twGWou54CqEexvgRC60CeJYxPTzVpYE7AM2GYLDSEM6kpR8Efabm8Fef+w6RLf0jP8csnwZdttrXW0wqtVchjOGYr5QIdBiMUvDfvMFfS5oaI0SJ7pzYC9q8G/6W1w0wLRTukmjFt+xOgB1m5HruZ4iPWKPiOSvEGqx27j50H/6i8v95D3+BkZusXPnk2IC8KhRKO9MPngbA/+qXL9efP958JNg1B7336Ajo4hgwP0QA9oY4xgErjrEL4aB19D9QSwMEFAAAAAgA+3teXQgOaXPXAwAAvAoAACoAAABUZXN0L3Rlc3RzL3Rlc3RfY29tcGFjdF9zY2VuZV90cmFuc3BvcnQucHm1Vs2TmzYUv/NXML6A1ioBs961cbhsZjM5JZ102wvDMDLIWWpAriRsb6d/fJ8QYGCzSWbTckDi6X383tP7YDabvWdpLWhmHjiTLGWFyekXToXIWWWecvnIamnm5YFxmVdfTHaQcEAK85g3HBk90CqjVZpT4cxmM0OzthJFvnVqmRfGjrPSPBD5CJT2zPwVPjt2IXmdyu6rrnIpqZCG8fnTp4dQMdpJsssLmiTIAXCsOFIbOQfCaSVFtIgNcaBpODbqKFqiLGvRgqVEgbetlJUHkspEpLSiieSkEkrQwsrcG+v+SIq6YX3zAqdzeLKQUbKsLujUqqZquwqBrV5oo95OwUhGuUPPAEzz2XpBhmGkBRHCfKct/qYMPkAMhN1Fw1Gf74igKDBMeDK6MwWVvx9sQYtdS1SP+nTSUGt2Rgo7B2zvFluua135C2x5XrNGD7ym8dVqOY/ek0KoLcDqLJ1yThtDOGU1BD38yCqK4QJIu2X7UCkY4GgZ9WLmO7PbCbNi0lRSJgVDZuTGV2u/l9NKm7eSajffEUrL0Pbevl0t0S+ekmJ7zeZe1GqOtf8iB6ey5pVpb58g3na0wHklbbZHMZprkr9od4663kd6ttvgobnOYEjJdG9bYOSDha+0v6jX3z2ttsYxNE9LR7JEkzy4jgKuu6AWmpMXDga3ovICsoxwQRN6hmue5kIa6jhtUljDm+XSX25IRyPRGojebc99Cpvcae46xQRtzpqQOlDhLKP2CY2zDDKWcnn/V00Ku6CVfUbYv4Vkn561WjLaaAGu09QJ6CEv+TBEFXk6Q8GBFY4WjR+vg+m+DufflLPkKJIdgZ6STaESvB2ARfiyh/Joquo7yMZ+EITwYv1VkY9sDLmXwGPCFn3D4tjdZ4J4O/UehkTTS0mVJUfK1QgQ0xhML2MQjwuUHeNmSQpYSpg9eWVG5yjwYtyW3jWU3DnyghgDeRG3BRN5K+w2J9dBz+rjVUNaBHEcjEpNTa+hz59JLkDiD+ju9J5zxlEwDkAP6ILzx3U89zYKloB2kMYKMeGcPI1isjnBAAu915vUgTg96wt1RY6QpWTbXhgM9q/d16BFrH8OyKB5oB/y21/exP+E3mL1Xzj/bXt9/Sm7Nzf/S8C3JOubcSU5/BJNY/1ISRaOR8XdHYwKHy+xd3vxoSRiL4adJPKvl0E8Kp/WRFM8ke1iD7sIt2uMI3vt91sXu2o3qY9zqOB0taW6T6tSDbwGwXxrWc6fLIe+NMb84Q7GG0Uah0LQSf5cAZ7VfxCM5iSpSAk/e2FoJUlJ8ipJrKD/D1IEGxn/AlBLAQIUAxQAAAAIAPt7Xl3BLQZbLAUAACkOAAAlAAAAAAAAAAAAAACAAQAAAABFdmFsdWF0aW9uL2NvbXBhY3Rfc2NlbmVfdHJhbnNwb3J0LnB5UEsBAhQDFAAAAAgA+3teXX028qLjCwAA9xwAAC8AAAAAAAAAAAAAAIABbwUAAGV4cGVyaW1lbnRzL3N0cmVhbV9zY2hlZHVsZXIvbWluaV9rYWdnbGVfcnVuLnB5UEsBAhQDFAAAAAgA+3teXQgOaXPXAwAAvAoAACoAAAAAAAAAAAAAAIABnxEAAFRlc3QvdGVzdHMvdGVzdF9jb21wYWN0X3NjZW5lX3RyYW5zcG9ydC5weVBLBQYAAAAAAwADAAgBAAC+FQAAAAA=')
    assert hashlib.sha256(blob).hexdigest()=='56e16045a2e54eca93c878d90ced61c64e1ae07e5669a98af52f30ce627cdb7f'
    SOURCE=RUN/'source'
    SOURCE.mkdir()
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        z.extractall(SOURCE)
    expected={'Evaluation/compact_scene_transport.py': '15a7f1e4aac1435761de5ee53f8cc245c223fb057d281f588928d999fb95caf9', 'experiments/stream_scheduler/mini_kaggle_run.py': '8e0b7a319c51b4c42ed349fd8af0a2a7b332a73f525b2eac4892563d56343203', 'Test/tests/test_compact_scene_transport.py': 'c13149ed9f7c0e33cf3c39a40a1b6c7cae6e11d524c014998668a84cd50fcc70'}
    for name,value in expected.items():
        assert hashlib.sha256((SOURCE/name).read_bytes()).hexdigest()==value
    manifest={'commit':'032c23a3dafc7bfeb39a84ba55193b6b6f3a22a0','dirty_snapshot':True,'source_hashes':expected,'input_kernel':'satyapaladuguplus/semantic-image-pool-scene-cache-20261030/5','expected_archive_sha256':'0f8c6186b09dcf6ab87911df642612919377157bb1585082bed1c93b1506fe2c','gpu_budget':0,'internal_driver_timeout_seconds':900}
    (RUN/'manifest.json').write_text(json.dumps(manifest,indent=2));print(json.dumps(manifest,indent=2),flush=True)


In [ ]:
with stage('03_input_and_regression'):
    matches=list(Path('/kaggle/input').rglob('Mini_dataset_6000.tar'))
    if len(matches)!=1:raise ValueError('Expected exactly one attached mini archive; found '+str(matches))
    ARCHIVE=matches[0];print('Input archive',ARCHIVE,'bytes',ARCHIVE.stat().st_size,flush=True)
    (RUN/'input.json').write_text(json.dumps({'path':str(ARCHIVE),'bytes':ARCHIVE.stat().st_size},indent=2))
    subprocess.run([sys.executable,'-m','unittest','discover','-s',str(SOURCE/'Test/tests'),'-p','test_compact_scene_transport.py','-v'],check=True,timeout=60)


In [ ]:
with stage('04_transport_and_dynamic_replay'):
    command=[sys.executable,'-u',str(SOURCE/'experiments/stream_scheduler/mini_kaggle_run.py'),'--archive',str(ARCHIVE),'--output',str(RUN/'results')]
    print('Command',command,flush=True)
    with (RUN/'logs/driver.log').open('w',buffering=1) as log:
        proc=subprocess.Popen(command,stdout=log,stderr=subprocess.STDOUT,text=True)
        try:
            code=proc.wait(timeout=900)
        except subprocess.TimeoutExpired:
            proc.kill();proc.wait();raise RuntimeError('Bounded CPU driver timed out; partial logs preserved')
    print((RUN/'logs/driver.log').read_text(),flush=True)
    if code:raise RuntimeError('Driver failed with exit '+str(code))
    summary=json.loads((RUN/'results/summary.json').read_text())
    assert summary['scene_roundtrips']==6000 and summary['channels_retained']==93
    for traces in summary['traces'].values():
        for result in traces.values():assert result['offered']==result['selected']+result['unselected']
    print(json.dumps({k:v for k,v in summary.items() if k!='traces'},indent=2),flush=True)


In [ ]:
with stage('05_report_and_recovery'):
    summary=json.loads((RUN/'results/summary.json').read_text())
    text='# Real-cache scheduler component result\n\n'+json.dumps({k:v for k,v in summary.items() if k!='traces'},indent=2)+'\n\nNo P/neural/RL training or independent urgency evaluation occurred. Four baseline policies across three dynamic MIXED_IID traces; delays stratified by input SPI only. Temporal missions/relevance/novelty and qualified teacher supervision remain the next components. This notebook did not alter image-cache or SPI production.\n'
    (RUN/'REPORT.md').write_text(text)
    paths={str(p.relative_to(RUN)):hashlib.sha256(p.read_bytes()).hexdigest() for p in RUN.rglob('*') if p.is_file() and p.name!='artifact_manifest.json' and '/logs/' not in str(p)}
    (RUN/'artifact_manifest.json').write_text(json.dumps(paths,indent=2))
    print('Saved results, traces, source, manifest and cell logs under',RUN,flush=True)
# Capture finished cell status/logs after stage exit in the recovery ZIP.
archive=shutil.make_archive(str(RUN.parent/(RUN.name+'-recovery')),'zip',RUN)
print('RECOVERY',archive,'sha256',hashlib.sha256(Path(archive).read_bytes()).hexdigest(),flush=True)
